# 06a: Generate Decoder Training Data

**Runtime: A100** (Mistral-7B needs the VRAM)

Uses Mistral-7B-Instruct as teacher to generate (routing_output, response) pairs.
Saves to Google Drive so notebook 06b can pick it up on a T4.

**Time:** ~20-30 min | **Cost:** $0

In [ ]:
!pip install -q sentence-transformers
!nvidia-smi | head -4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/HLC')

In [ ]:
from pathlib import Path
from hlc.config import Config
from hlc.decoder_training import DecoderTrainer
from experiments.seed_basic import BASIC_FACTS

config = Config()
trainer = DecoderTrainer(config)

DATA_DIR = Path('/content/drive/MyDrive/HLC/data')
BASE_DATASET = DATA_DIR / 'decoder_base.jsonl'
TRAINING_DATA = DATA_DIR / 'decoder_training.jsonl'

print(f'Facts: {len(BASIC_FACTS)}')
print(f'Teacher: {config.lm_model_name}')

## Step 1: Generate base pairs via Mistral-7B (~20-30 min)

In [ ]:
base_examples = trainer.generate_base_dataset(
    facts=BASIC_FACTS,
    output_path=BASE_DATASET,
    verbose=True,
)
print(f'\nBase dataset: {len(base_examples)} examples')

In [ ]:
# Preview
for i, ex in enumerate(base_examples[:5]):
    print(f'--- Example {i+1} ---')
    print(f'Q: {ex.query}')
    print(f'K: {ex.knowledge[0][:80]}...')
    print(f'R: {ex.response}')
    print()

## Step 2: Augment to 10K (instant, no GPU)

In [ ]:
augmented = trainer.augment_dataset(base_examples, target_size=10000)
trainer.prepare_training_data(augmented, TRAINING_DATA)

print(f'Augmented: {len(augmented)} examples')
print(f'Saved to: {TRAINING_DATA}')
print(f'File size: {TRAINING_DATA.stat().st_size / 1024 / 1024:.1f} MB')
print()
print('--- Sample ---')
print(augmented[0].to_training_text())

## Done

Training data is saved to Google Drive at:
- `data/decoder_base.jsonl` — base examples
- `data/decoder_training.jsonl` — augmented 10K training set

**Next:** Switch to T4 runtime and open `06b_train_decoder.ipynb`